In [5]:
import import_before_profile

In [6]:
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from jaxtyping import Array
from tensorflow_probability.substrates import jax as tfp

from qdots_qll.data import Data
from qdots_qll.distributions import (
    Distribution,
    update_log_weights,
)
from qdots_qll.exp_design import (
    GameExpDesignFullAdaptivity,
    GameExpDesignTimeAdaptivity,
    RandExpDesignGAME,
    TraceGameExpDesignFullAdaptivity,
)
from qdots_qll.experiments import ExperimentSingleDotWeakCouplingGAME
from qdots_qll.models.single_dot_weak_coupling_GAME import (
    SingleDotWeakCouplingGAME,
)
from qdots_qll.resamplers import LiuWestResampler, MetropolisSampler
from qdots_qll.smc import SMCUpdater, replace_single_datum, replace_single_exp


def _do_not_resample(key, dist: Distribution, *args, **kwargs):
    return key, dist


def _resample(key, dist, iteration, data, resampler, *args, **kwargs):
    key, subkey = jax.random.split(key)
    new_dist: Distribution = resampler.resample(
        subkey=subkey,
        index_data=iteration,
        distribution=dist,
        data=data,
    )
    return key, new_dist


@jax.jit
def step_time_adaptivity(
    key: Array,
    iteration: int,
    distribution: Distribution,
    data: Data,
    prob_initial_state,
    prob_measurement_basis,
    *args,
    **kwargs,
) -> tuple[Array, Distribution]:
    # generate experiment
    # Measure experiment
    # create datum
    # append to data
    # update distribution
    # resample if necessary
    key, subkey = jax.random.split(key)
    # experiment = e1
    (experiment,) = exp_design.generate_experiment(
        model=model,
        distribution=distribution,
        data=data,
        subkey=subkey,
        prob_initial_state=prob_initial_state,
        prob_measurement_basis=prob_measurement_basis,
    )

    key, subkey = jax.random.split(key)

    outcome = model.measure_one_experiment(subkey, experiment)
    datum = Data(experiment, outcome)

    log_lkl = model.log_lkl_datum_multiple_particles(
        distribution.particles_locations, datum
    )

    data = replace_single_datum(data, datum, iteration)

    distribution: Distribution = update_log_weights(
        dist=distribution, new_log_lkl=log_lkl
    )

    key, subkey = jax.random.split(key)
    key, distribution = jax.lax.cond(
        distribution.check_resampling(),
        _resample,
        _do_not_resample,
        *(key, distribution, iteration, data, resampler),
    )

    iteration = iteration + 1

    prob_initial_state = prob_initial_state
    prob_measurement_basis = prob_measurement_basis
    return (
        key,
        iteration,
        distribution,
        data,
        prob_initial_state,
        prob_measurement_basis,
    )


@jax.jit
def step_full_time_adaptivity(
    key: Array,
    iteration: int,
    distribution: Distribution,
    data: Data,
    prob_initial_state,
    prob_measurement_basis,
    *args,
    **kwargs,
) -> tuple[Array, Distribution]:
    # generate experiment
    # Measure experiment
    # create datum
    # append to data
    # update distribution
    # resample if necessary
    key, subkey = jax.random.split(key)
    # experiment = e1
    experiment, opt_prob_initial_state, opt_prob_measurement_basis = (
        exp_design.generate_experiment(
            model=model,
            distribution=distribution,
            data=data,
            subkey=subkey,
            prob_initial_state=prob_initial_state,
            prob_measurement_basis=prob_measurement_basis,
        )
    )

    key, subkey = jax.random.split(key)

    outcome = model.measure_one_experiment(subkey, experiment)
    datum = Data(experiment, outcome)

    log_lkl = model.log_lkl_datum_multiple_particles(
        distribution.particles_locations, datum
    )

    data = replace_single_datum(data, datum, iteration)

    distribution: Distribution = update_log_weights(
        dist=distribution, new_log_lkl=log_lkl
    )

    key, subkey = jax.random.split(key)
    key, distribution = jax.lax.cond(
        distribution.check_resampling(),
        _resample,
        _do_not_resample,
        *(key, distribution, iteration, data, resampler),
    )

    iteration = iteration + 1

    prob_initial_state = opt_prob_initial_state
    prob_measurement_basis = opt_prob_measurement_basis
    return (
        key,
        iteration,
        distribution,
        data,
        prob_initial_state,
        prob_measurement_basis,
    )

In [4]:
no_particles = 500
N_max_exps = 10000

In [6]:
model = SingleDotWeakCouplingGAME()

key = jax.random.key(0)
key, subkey = jax.random.split(key)


boundaries = jnp.array(
    [
        [0.1, 0.5],
        [0.1, 0.5],
        [0.01, 0.2],
        [-0.5, -0.01],
    ]
)


loc = boundaries.mean(axis=1)
scale = boundaries.std(axis=1)

truncated_norm = tfp.distributions.TruncatedNormal(
    loc=loc, scale=scale / 1.5, low=boundaries[:, 0], high=boundaries[:, 1]
)

init_particles_locations = truncated_norm.sample(
    seed=subkey, sample_shape=(no_particles,)
)
weights = jnp.ones(no_particles) / no_particles

dist = Distribution(particles_locations=init_particles_locations, weights=weights)


_aux_exps = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([-99, -99, -99])[None, :], N_max_exps, axis=0)
)

_aux_outcomes = jnp.ones(N_max_exps) * 0

aux_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    _aux_exps, _aux_outcomes
)

data = aux_data

prob_initial_state = jnp.ones(4) / 4
prob_measurement_basis = jnp.ones(3) / 3

In [8]:
exp_design = GameExpDesignTimeAdaptivity(0.01, 40.0)
resampler = LiuWestResampler(
    boundaries,
)
smc = SMCUpdater(model, exp_design, resampler)

In [9]:
model = SingleDotWeakCouplingGAME()

key = jax.random.key(0)
key, subkey = jax.random.split(key)
boundaries = jnp.array(
    [
        [0.1, 0.5],
        [0.1, 0.5],
        [0.01, 0.2],
        [-0.5, -0.01],
    ]
)
loc = boundaries.mean(axis=1)
scale = boundaries.std(axis=1)


_aux_exps = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([-99, -99, -99])[None, :], N_max_exps, axis=0)
)

_aux_outcomes = jnp.ones(N_max_exps) * 0

aux_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    _aux_exps, _aux_outcomes
)


def initialize_distributions(subkey):
    key, subkey = jax.random.split(subkey)
    truncated_norm = tfp.distributions.TruncatedNormal(
        loc=loc, scale=scale / 1.5, low=boundaries[:, 0], high=boundaries[:, 1]
    )

    init_particles_locations = truncated_norm.sample(
        seed=subkey, sample_shape=(no_particles,)
    )
    weights = jnp.ones(no_particles) / no_particles

    dist = Distribution(particles_locations=init_particles_locations, weights=weights)
    iteration = 0
    prob_initial_state = jnp.ones(4) / 4
    prob_measurement_basis = jnp.ones(3) / 3
    return key, iteration, dist, aux_data, prob_initial_state, prob_measurement_basis


@jax.jit
def replace_log_arrays(
    inputs, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
):
    keys, iterations, dists, datas, prob_initial_states, prob_measurement_basiss = (
        inputs
    )

    iteration = iterations[0]
    covariances_arr = covariances_arr.at[iteration].set(
        jax.vmap(lambda x: x.cov())(dists)
    )
    e_values_arr = e_values_arr.at[iteration].set(jax.vmap(lambda x: x.ev())(dists))
    pr_rho0_arr = pr_rho0_arr.at[iteration].set(prob_initial_states)
    pr_basis_arr = pr_basis_arr.at[iteration].set(prob_measurement_basiss)

    return covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr


jitted_vmap_step_full_opt = jax.jit(
    jax.vmap(step_full_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)
jitted_vmap_step_time_opt = jax.jit(
    jax.vmap(step_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)


# Only time adaptivity

In [10]:
no_runs = 100
iter_max = 4000
subkeys = jax.random.split(key, no_runs + 1)
key = subkeys[0]
subkeys = subkeys[1:]

covariances_arr = jnp.zeros([N_max_exps, no_runs, 4, 4])
e_values_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_rho0_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_basis_arr = jnp.zeros([N_max_exps, no_runs, 3])


carry_args = jax.vmap(initialize_distributions)(subkeys)
for i in range(iter_max):
    carry_args = jitted_vmap_step_time_opt(*carry_args)
    if i % 25 == 0:
        print(f"Iteration number: {i}")
    covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr = replace_log_arrays(
        carry_args, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
    )

DEBUG:2024-09-14 15:36:55,449:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-14 15:36:55,450:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-14 15:36:55,452:jax._src.lru_cache:107: Cache hit for key: 'jit__threefry_split-128ef2d1c8ec0bf1b512b5d1b5b8557b7be4e3d274b7f26079b0169cb793cc87'
DEBUG:2024-09-14 15:36:55,457:jax._src.compiler:98: Persistent compilation cache hit for 'jit__threefry_split' with key 'jit__threefry_split-128ef2d1c8ec0bf1b512b5d1b5b8557b7be4e3d274b7f26079b0169cb793cc87'
DEBUG:2024-09-14 15:36:55,466:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-14 15:36:55,467:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-14 15:36:55,468:jax._src.lru_cache:107: Cache hit for key: 'jit_dyn

Iteration number: 0
Iteration number: 25
Iteration number: 50
Iteration number: 75
Iteration number: 100
Iteration number: 125
Iteration number: 150
Iteration number: 175
Iteration number: 200
Iteration number: 225
Iteration number: 250
Iteration number: 275
Iteration number: 300
Iteration number: 325
Iteration number: 350
Iteration number: 375
Iteration number: 400
Iteration number: 425
Iteration number: 450
Iteration number: 475
Iteration number: 500
Iteration number: 525
Iteration number: 550
Iteration number: 575
Iteration number: 600
Iteration number: 625
Iteration number: 650
Iteration number: 675
Iteration number: 700
Iteration number: 725
Iteration number: 750
Iteration number: 775
Iteration number: 800
Iteration number: 825
Iteration number: 850
Iteration number: 875
Iteration number: 900
Iteration number: 925
Iteration number: 950
Iteration number: 975
Iteration number: 1000
Iteration number: 1025
Iteration number: 1050
Iteration number: 1075
Iteration number: 1100
Iteration 

In [11]:
import joblib

joblib.dump(
    [covariances_arr[:], e_values_arr[:], pr_rho0_arr[:], pr_basis_arr[:], carry_args],
    filename="results_single_dot_lately/only_time_opt.job",
)


['results_single_dot_lately/only_time_opt.job']

# No adaptivity

In [13]:
exp_design = RandExpDesignGAME()
resampler = LiuWestResampler(
    boundaries,
)
model = SingleDotWeakCouplingGAME()
smc = SMCUpdater(model, exp_design, resampler)


key = jax.random.key(0)
key, subkey = jax.random.split(key)
boundaries = jnp.array(
    [
        [0.1, 0.5],
        [0.1, 0.5],
        [0.01, 0.2],
        [-0.5, -0.01],
    ]
)
loc = boundaries.mean(axis=1)
scale = boundaries.std(axis=1)


_aux_exps = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([-99, -99, -99])[None, :], N_max_exps, axis=0)
)

_aux_outcomes = jnp.ones(N_max_exps) * 0

aux_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    _aux_exps, _aux_outcomes
)


def initialize_distributions(subkey):
    key, subkey = jax.random.split(subkey)
    truncated_norm = tfp.distributions.TruncatedNormal(
        loc=loc, scale=scale / 1.5, low=boundaries[:, 0], high=boundaries[:, 1]
    )

    init_particles_locations = truncated_norm.sample(
        seed=subkey, sample_shape=(no_particles,)
    )
    weights = jnp.ones(no_particles) / no_particles

    dist = Distribution(particles_locations=init_particles_locations, weights=weights)
    iteration = 0
    prob_initial_state = jnp.ones(4) / 4
    prob_measurement_basis = jnp.ones(3) / 3
    return key, iteration, dist, aux_data, prob_initial_state, prob_measurement_basis


@jax.jit
def replace_log_arrays(
    inputs, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
):
    keys, iterations, dists, datas, prob_initial_states, prob_measurement_basiss = (
        inputs
    )

    iteration = iterations[0]
    covariances_arr = covariances_arr.at[iteration].set(
        jax.vmap(lambda x: x.cov())(dists)
    )
    e_values_arr = e_values_arr.at[iteration].set(jax.vmap(lambda x: x.ev())(dists))
    pr_rho0_arr = pr_rho0_arr.at[iteration].set(prob_initial_states)
    pr_basis_arr = pr_basis_arr.at[iteration].set(prob_measurement_basiss)

    return covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr


jitted_vmap_step_full_opt = jax.jit(
    jax.vmap(step_full_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)
jitted_vmap_step_time_opt = jax.jit(
    jax.vmap(step_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)


no_runs = 100
iter_max = 4000
subkeys = jax.random.split(key, no_runs + 1)
key = subkeys[0]
subkeys = subkeys[1:]

covariances_arr = jnp.zeros([N_max_exps, no_runs, 4, 4])
e_values_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_rho0_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_basis_arr = jnp.zeros([N_max_exps, no_runs, 3])


carry_args = jax.vmap(initialize_distributions)(subkeys)
for i in range(iter_max):
    carry_args = jitted_vmap_step_time_opt(*carry_args)
    if i % 25 == 0:
        print(f"Iteration number: {i}")
    covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr = replace_log_arrays(
        carry_args, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
    )


import joblib

joblib.dump(
    [covariances_arr[:], e_values_arr[:], pr_rho0_arr[:], pr_basis_arr[:], carry_args],
    filename="results_single_dot_lately/rand_time.job",
)


DEBUG:2024-09-14 17:57:03,020:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-14 17:57:03,021:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-14 17:57:03,048:jax._src.lru_cache:107: Cache hit for key: 'jit_step_time_adaptivity-1a63b75919221c4d1b6726d4e7091676c80f489d7c74862385f5bf27d7015c4b'
DEBUG:2024-09-14 17:57:03,513:jax._src.compiler:98: Persistent compilation cache hit for 'jit_step_time_adaptivity' with key 'jit_step_time_adaptivity-1a63b75919221c4d1b6726d4e7091676c80f489d7c74862385f5bf27d7015c4b'
DEBUG:2024-09-14 17:57:06,306:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-14 17:57:06,307:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-14 17:57:06,309:jax._src.lru_cache:107: Cache hit fo

Iteration number: 0
Iteration number: 25
Iteration number: 50
Iteration number: 75
Iteration number: 100
Iteration number: 125
Iteration number: 150
Iteration number: 175
Iteration number: 200
Iteration number: 225
Iteration number: 250
Iteration number: 275
Iteration number: 300
Iteration number: 325
Iteration number: 350
Iteration number: 375
Iteration number: 400
Iteration number: 425
Iteration number: 450
Iteration number: 475
Iteration number: 500
Iteration number: 525
Iteration number: 550
Iteration number: 575
Iteration number: 600
Iteration number: 625
Iteration number: 650
Iteration number: 675
Iteration number: 700
Iteration number: 725
Iteration number: 750
Iteration number: 775
Iteration number: 800
Iteration number: 825
Iteration number: 850
Iteration number: 875
Iteration number: 900
Iteration number: 925
Iteration number: 950
Iteration number: 975
Iteration number: 1000
Iteration number: 1025
Iteration number: 1050
Iteration number: 1075
Iteration number: 1100
Iteration 

['results_single_dot_lately/rand_time.job']

# All adaptivity but with the trace

In [11]:
no_particles = 500
N_max_exps = 10000

boundaries = jnp.array(
    [
        [0.1, 0.5],
        [0.1, 0.5],
        [0.01, 0.2],
        [-0.5, -0.01],
    ]
)


exp_design = TraceGameExpDesignFullAdaptivity(0.01, 40.0)
resampler = LiuWestResampler(
    boundaries,
)
model = SingleDotWeakCouplingGAME()
smc = SMCUpdater(model, exp_design, resampler)


key = jax.random.key(0)
key, subkey = jax.random.split(key)
loc = boundaries.mean(axis=1)
scale = boundaries.std(axis=1)


_aux_exps = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([-99, -99, -99])[None, :], N_max_exps, axis=0)
)

_aux_outcomes = jnp.ones(N_max_exps) * 0

aux_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    _aux_exps, _aux_outcomes
)


def initialize_distributions(subkey):
    key, subkey = jax.random.split(subkey)
    truncated_norm = tfp.distributions.TruncatedNormal(
        loc=loc, scale=scale / 1.5, low=boundaries[:, 0], high=boundaries[:, 1]
    )

    init_particles_locations = truncated_norm.sample(
        seed=subkey, sample_shape=(no_particles,)
    )
    weights = jnp.ones(no_particles) / no_particles

    dist = Distribution(particles_locations=init_particles_locations, weights=weights)
    iteration = 0
    prob_initial_state = jnp.ones(4) / 4
    prob_measurement_basis = jnp.ones(3) / 3
    return key, iteration, dist, aux_data, prob_initial_state, prob_measurement_basis


@jax.jit
def replace_log_arrays(
    index, inputs, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
):
    keys, iterations, dists, datas, prob_initial_states, prob_measurement_basiss = (
        inputs
    )

    # iteration = iterations[0]
    iteration = index
    covariances_arr = covariances_arr.at[iteration].set(
        jax.vmap(lambda x: x.cov())(dists)
    )
    e_values_arr = e_values_arr.at[iteration].set(jax.vmap(lambda x: x.ev())(dists))
    pr_rho0_arr = pr_rho0_arr.at[iteration].set(prob_initial_states)
    pr_basis_arr = pr_basis_arr.at[iteration].set(prob_measurement_basiss)

    return covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr


jitted_vmap_step_full_opt = jax.jit(
    jax.vmap(step_full_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)
jitted_vmap_step_time_opt = jax.jit(
    jax.vmap(step_time_adaptivity, in_axes=(0, 0, 0, 0, 0, 0))
)


no_runs = 100
iter_max = 4000
subkeys = jax.random.split(key, no_runs + 1)
key = subkeys[0]
subkeys = subkeys[1:]

covariances_arr = jnp.zeros([N_max_exps, no_runs, 4, 4])
e_values_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_rho0_arr = jnp.zeros([N_max_exps, no_runs, 4])
pr_basis_arr = jnp.zeros([N_max_exps, no_runs, 3])


carry_args = jax.vmap(initialize_distributions)(subkeys)
for i in range(iter_max):
    carry_args = jitted_vmap_step_full_opt(*carry_args)
    if i % 25 == 0:
        print(f"Iteration number: {i}")
    covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr = replace_log_arrays(
        i, carry_args, covariances_arr, e_values_arr, pr_rho0_arr, pr_basis_arr
    )


import joblib

joblib.dump(
    [covariances_arr[:], e_values_arr[:], pr_rho0_arr[:], pr_basis_arr[:], carry_args],
    filename="results_single_dot_lately/full_opt_trace.job",
)


DEBUG:2024-09-17 16:44:00,602:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-17 16:44:00,603:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-17 16:44:00,635:jax._src.lru_cache:104: Cache miss for key: 'jit_step_full_time_adaptivity-5972e56735f0a9e15ec2ae8eb80a909763bb406c10810c00dadd9c316d5a0eb3'
DEBUG:2024-09-17 16:44:20,309:jax._src.compiler:704: 'jit_step_full_time_adaptivity' took at least 0.00 seconds to compile (19.67s)
DEBUG:2024-09-17 16:44:22,172:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-17 16:44:22,172:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-17 16:44:22,173:jax._src.lru_cache:104: Cache miss for key: 'jit_replace_log_arrays-e4e947f0cb42816c7b2adc2fbf2f71008c77adf6d5ac7dc

Iteration number: 0
Iteration number: 25
Iteration number: 50
Iteration number: 75
Iteration number: 100
Iteration number: 125
Iteration number: 150
Iteration number: 175
Iteration number: 200
Iteration number: 225
Iteration number: 250
Iteration number: 275
Iteration number: 300
Iteration number: 325
Iteration number: 350
Iteration number: 375
Iteration number: 400
Iteration number: 425
Iteration number: 450
Iteration number: 475
Iteration number: 500
Iteration number: 525
Iteration number: 550
Iteration number: 575
Iteration number: 600
Iteration number: 625
Iteration number: 650
Iteration number: 675
Iteration number: 700
Iteration number: 725
Iteration number: 750
Iteration number: 775
Iteration number: 800
Iteration number: 825
Iteration number: 850
Iteration number: 875
Iteration number: 900
Iteration number: 925
Iteration number: 950
Iteration number: 975
Iteration number: 1000
Iteration number: 1025
Iteration number: 1050
Iteration number: 1075
Iteration number: 1100
Iteration 

['results_single_dot_lately/full_opt_trace.job']